# Process mining: восстановление процесса возврата по журналу событий

Process mining восстанавливает **реальный** процесс из логов информационных систем и сравнивает его с тем, как процесс задуман. Это показывает, где на практике всё идёт не по регламенту.

Беру тот же процесс возврата, что в [кейсе 1](../../project-1-process-optimization/), но теперь не рисую его руками, а восстанавливаю из журнала событий через PM4Py и ищу отклонения.

Журнал синтетический и генерируется ниже с фиксированным сидом. Эталон (как должно быть): *Создана заявка → Проверка права → Накладная → Товар сдан → Товар проверен → Возврат денег*.


## 1. Генерация журнала событий

В лог заложено несколько вариантов поведения: основной путь, отказы, повторная проверка товара и — главное — случаи, где пропущена проверка права на возврат (нарушение регламента).


In [ ]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta

rng = np.random.default_rng(42)

happy  = ['Создана заявка','Проверка права','Накладная','Товар сдан','Товар проверен','Возврат денег']
reject = ['Создана заявка','Проверка права','Отказ']
rework = ['Создана заявка','Проверка права','Накладная','Товар сдан','Товар проверен','Повторная проверка','Товар проверен','Возврат денег']
skip   = ['Создана заявка','Накладная','Товар сдан','Товар проверен','Возврат денег']  # пропущена проверка права

variants = [happy, reject, rework, skip]
weights  = [0.70, 0.10, 0.12, 0.08]

rows = []
start = datetime(2024, 1, 1)
for case in range(1, 801):
    path = variants[rng.choice(len(variants), p=weights)]
    t = start + timedelta(days=int(rng.integers(0, 300)), hours=int(rng.integers(0, 8)))
    for act in path:
        rows.append({'case_id': f'R{case:04d}', 'activity': act, 'timestamp': t})
        t = t + timedelta(hours=float(rng.uniform(1, 48)))

df = pd.DataFrame(rows)
print('Событий:', len(df), '| Заявок:', df.case_id.nunique())
df.head(8)


## 2. Обзор вариантов

Сколько реальных вариантов процесса встречается в логе и как часто.


In [ ]:
import pm4py

log = pm4py.format_dataframe(df, case_id='case_id',
                            activity_key='activity', timestamp_key='timestamp')

variants_count = pm4py.get_variants(log)
print('Уникальных вариантов процесса:', len(variants_count))
for var, cnt in sorted(variants_count.items(), key=lambda x: -len(x[1]) if isinstance(x[1], list) else -x[1])[:6]:
    n = len(cnt) if isinstance(cnt, list) else cnt
    print(n, '|', ' -> '.join(var) if isinstance(var, tuple) else var)


## 3. Восстановление модели (discovery)

Inductive miner строит модель процесса по логу. Её можно экспортировать картинкой (нужен graphviz).


In [ ]:
net, im, fm = pm4py.discover_petri_net_inductive(log)
print('Переходов в сети:', len(net.transitions), '| мест:', len(net.places))

# Визуализация (раскомментируйте, если установлен graphviz):
# pm4py.save_vis_petri_net(net, im, fm, 'discovered_process.png')


## 4. Проверка соответствия (conformance)

Сравниваю лог с эталонной моделью и ищу, где реальность расходится с регламентом. Отдельно выделяю случаи с пропущенной проверкой права — это контрольный риск.


In [ ]:
fitness = pm4py.fitness_token_based_replay(log, net, im, fm)
print('Fitness (насколько лог укладывается в модель):',
      round(fitness['average_trace_fitness'], 3))

# Доля заявок с пропущенной проверкой права
def has_check(group):
    return 'Проверка права' in set(group['activity'])

checked = df.groupby('case_id').apply(has_check, include_groups=False)
skipped = (~checked).mean()
print('Доля заявок без проверки права:', round(float(skipped), 3))


## 5. Выводы

Что показал анализ лога:

- **Реальный процесс сложнее нарисованного.** Кроме основного пути в логе есть отказы, повторные проверки товара и заявки без проверки права.
- **Нашлось контрольное нарушение:** примерно у каждой двенадцатой заявки пропущен шаг «Проверка права на возврат». Это прямой риск: возвраты уходят без проверки.
- **Повторная проверка товара** — петля, которая удлиняет процесс; стоит разобраться, почему товар проверяют дважды.

Что дальше: по таким находкам ставятся контроли в системе (нельзя двигаться дальше без проверки права) и считается, сколько времени и денег теряется на петлях. Process mining здесь нужен не ради модели, а чтобы увидеть процесс таким, какой он есть, а не каким его считают.
